<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0">

# European Derivatives Valuation

This part introduces into the modeling and valuation of European derivatives instruments (contingent claims) based on a single risk factor (e.g. a stock price, stock index level or interest rate). It also shows how to model and value portfolios composed of such instruments.

In [ ]:
!git clone https://github.com/tpq-classes/computational_finance_dx.git
import sys
sys.path.append('computational_finance_dx')

!git clone https://github.com/yhilpisch/dx.git
sys.path.append('dx')


In [ ]:
from dx import *
from pylab import plt
plt.style.use('seaborn-v0_8')

## Modeling the Risk Factor

Before moving on to the valuation classes, we need to model an instantiate an underlying risk factor, in this case a `geometric_brownian_motion` object. Background information is provided in the respective part of the documentation about model classes.

In [ ]:
r = constant_short_rate('r', 0.06)

In [ ]:
me = market_environment('me', dt.datetime(2020, 1, 1))

In [ ]:
me.add_constant('initial_value', 36.)
me.add_constant('volatility', 0.2)
me.add_constant('final_date', dt.datetime(2022, 12, 31))
me.add_constant('currency', 'EUR')
me.add_constant('frequency', 'W')
me.add_constant('paths', 75000)

In [ ]:
me.add_curve('discount_curve', r)

In [ ]:
gbm = geometric_brownian_motion('gbm', me)

## valuation_mcs_european_single

The first instrument we value is a European call option written on the single relevant risk factor as embodied by the `gbm` model object. To this end, we add a **maturity date** to the market environment and a **strike price**.

In [ ]:
me.add_constant('maturity', dt.datetime(2021, 12, 31))
me.add_constant('strike', 40.)

To instantiate a the `valuation_mcs_european_single` class, the following information/data is to be provided:

* name as a `string` object
* instance of a model class
* market environment
* payoff of the instrument a `string` object and containing "regular" Python/NumPy code

In [ ]:
call_eur = valuation_mcs_european_single(
                name='call_eur',
                underlying=gbm,
                mar_env=me,
                payoff_func='np.maximum(maturity_value - strike, 0)')

In this case, the payoff is that of a regular, plain vanilla European call option. If $T$ is the maturity date, $S_T$ the value of the relevant risk factor at that date and $K$ the strike price, the payoff $h_T$ at maturity of such an option is given by

$$
h_T = \max[S_T - K, 0]
$$

`maturity_value` represents the value vector of the risk factor at maturity. Any other "sensible" payoff definition is possible. For instance, the following works as well:

In [ ]:
payoff = 'np.maximum(np.minimum(maturity_value) * 2 - 50, 0)'

Other standardized payoff elemenets include `mean_value, max_value` and `min_value` representing maturity value vectors with the pathwise means, maxima and minima. Using these payoff elements allows the easy definition of options with **Asian features**.


Having instantiated the valuation class, the `present_value` method returns the present value Monte Carlo estimator for the call option.

In [ ]:
call_eur.present_value()

Similarly, the `delta` and `vega` methods return the delta and the vega of the option, estimated numerically by a forward difference scheme and Monte Carlo simulation.

In [ ]:
call_eur.delta()

In [ ]:
call_eur.vega()

This approach allows to work with such a valuation object similar to an analytical valuation formula like the one of Black-Scholes-Merton (1973). For example, you can **estimate and plot present values, deltas, gammas, vegas, thetas and rhos** for a range of different initial values of the risk factor.

In [ ]:
%%time
k_list = np.arange(12., 80.1, 2)
pv = []; de = []; ve = []; th = []; rh = []; ga = []
for k in k_list:
    call_eur.update(strike=k)
    pv.append(call_eur.present_value())
    de.append(call_eur.delta(0.5))
    ga.append(call_eur.gamma(2.0))
    ve.append(call_eur.vega(0.2))
    th.append(call_eur.theta())
    rh.append(call_eur.rho())

In [ ]:
%matplotlib inline

There is a little plot helper function available to plot these statistics conveniently.

In [ ]:
plot_option_stats_full(k_list, pv, de, ga, ve, th, rh)

In [ ]:
# call_eur.gamma?

In [ ]:
call_eur.gamma(interval=2, accuracy=7)

The second derivative we value is a **European put option**.

In [ ]:
me.add_constant('initial_value', 36.)
  # reset initial_value
me.add_constant('maturity', dt.datetime(2022, 12, 31))
  # longer maturity

In [ ]:
put_eur = valuation_mcs_european_single(
                name='put_eur',
                underlying=gbm,
                mar_env=me,
                payoff_func='np.maximum(strike - maturity_value, 0)')

In [ ]:
put_eur.present_value()

Since DX Analytics relies on Monte Carlo simulation and other *numerical* methods, the calculation of the delta and vega of such an option is identical to the call option case.

In [ ]:
put_eur.delta()

In [ ]:
put_eur.vega()

In [ ]:
%%time
k_list = np.arange(26., 46.1, 2.)
pv = []; de = []; ve = []
for k in k_list:
    put_eur.update(strike=k)
    pv.append(put_eur.present_value())
    de.append(put_eur.delta(0.5))
    ve.append(put_eur.vega(0.2))

In [ ]:
plot_option_stats(k_list, pv, de, ve)

## Portfolio Valuation

In general, market players (asset managers, investment banks, hedge funds, insurance companies, etc.) have to value not only single derivatvies instruments but rather portfolios composed of several derivatives instruments. A consistent derivatives portfolio valuation is particularly important when there are multiple derivatives written on the same risk factor and/or correlations between different risk factors.

These are the classes availble for a consistent portfolio valuation:

* `derivatives_position` to model a portfolio position
* `derivatives_portfolio` to model a derivatives portfolio

### derivatives_position

We work with the `market_environment` object from before and add information about the risk factor model we are using.

In [ ]:
me.add_constant('model', 'gbm')

A derivatives position consists of "data only" and not instantiated model or valuation objects. The necessary model and valuation objects are instantiated during the portfolio valuation.

In [ ]:
put = derivatives_position(
                name='put',  # name of position
                quantity=1,  # number of instruments
                underlyings=['gbm'],  # relevant risk factors
                mar_env=me,  # market environment
                otype='European single',  # the option type
                payoff_func='np.maximum(40. - maturity_value, 0)')
                    # the payoff funtion

The method `get_info` prints an overview of the all relevant information stored for the respective `derivatives_position` object.

In [ ]:
put.get_info()

### derivatives_portfolio

The `derivatives_portfolio` class implements the core portfolio valuation tasks. This sub-section illustrates to cases, one with uncorrelated underlyings and another one with correlated underlyings

#### Uncorrelated Underlyings

The first example is based on a portfolio with two single-risk factor instruments on **two different risk factors which are not correlated**. In addition to the `gbm` object, we define a `jump_diffusion` object.

In [ ]:
me_jump = market_environment('me_jump', dt.datetime(2020, 1, 1))

In [ ]:
me_jump.add_environment(me)
me_jump.add_constant('lambda', 0.6)
me_jump.add_constant('mu', -0.8)
me_jump.add_constant('delta', 0.1)
me_jump.add_constant('model', 'jd')

Based on this new risk factor model object, a European call option is defined.

In [ ]:
call_jump = derivatives_position(
                name='call_jump',
                quantity=3,
                underlyings=['jd'],
                mar_env=me_jump,
                otype='European single',
                payoff_func='np.maximum(maturity_value - 36., 0)')

Our **relevant market** now takes on the following form (defined a `dictionary` objects):

In [ ]:
risk_factors = {'gbm': me, 'jd': me_jump}
positions = {'put': put, 'call_jump': call_jump}

To instantiate the `derivatives_portfolio` class, a **valuation environment** (instance of `market_environment` class) is needed.

In [ ]:
val_env = market_environment('general', dt.datetime(2020, 1, 1))
val_env.add_constant('frequency', 'M')
val_env.add_constant('paths', 50000)
val_env.add_constant('starting_date', val_env.pricing_date)
val_env.add_constant('final_date', val_env.pricing_date)
val_env.add_curve('discount_curve', r)

For the **instantiation**, we pass all the elements to the portfolio class.

In [ ]:
port = derivatives_portfolio(
            name='portfolio',  # name
            positions=positions,  # derivatives positions
            val_env=val_env,  # valuation environment
            risk_factors=risk_factors, # relevant risk factors
            correlations=False,  # correlation between risk factors
            fixed_seed=False,  # fixed seed for randon number generation
            parallel=False)  # parallel valuation of portfolio positions

Once instantiated, the method `get_statistics` provides major portfolio statistics like position values, position deltas ans position vegas.

In [ ]:
%%time
stats = port.get_statistics()

In [ ]:
stats

The method returns a standard pandas `DataFrame` object with which you can work as you are used to.

In [ ]:
stats[['pos_value', 'pos_delta', 'pos_vega']].sum()

The metod `get_values` only calculates the present values of the derivatives instruments and positions and is therefore a bit less compute and time intensive.

In [ ]:
%time port.get_values()

The method `get_positions` provides detailed information about the single derivatives positions of the `derivatives_portfolio` object.

In [ ]:
port.get_positions()

Let us pick **two specific simulated paths**, one for each risk factor, and let us visualize these.

In [ ]:
path_no = 0
paths1 = port.underlying_objects['gbm'].get_instrument_values()[:, path_no]
paths2 = port.underlying_objects['jd'].get_instrument_values()[:, path_no]

The plot illustrates that the two paths are indeed **not "really" correlated**. However, in this case a **large jump** occurs for the `jump_diffusion` object.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(port.time_grid, paths1, 'r', label='gbm')
plt.plot(port.time_grid, paths2, 'b', label='jd')
plt.gcf().autofmt_xdate()
plt.legend(loc=0); plt.grid(True)
# non-correlated underlyings
# -- with a larger jump for one risk factor

#### Correlated Underlyings

The second example case is exactly the same but now with a **highly positive correlation** between the two relevant risk factors. Correlations are to be provided as a `list`  of `list` objects using the risk factor model names to reference them.

In [ ]:
correlations = [['gbm', 'jd', 0.9]]

Except from now passing this new object, the application and usage remains the same.

In [ ]:
port = derivatives_portfolio(
            name='portfolio',
            positions=positions,
            val_env=val_env,
            risk_factors=risk_factors,
            correlations=correlations,
            fixed_seed=True,
            parallel=False)

In [ ]:
port.get_statistics()

The **Cholesky matrix** has been added to the valuation environment (which gets passed to the risk factor model objects).

In [ ]:
port.val_env.lists['cholesky_matrix']

Let us pick **two specific simulated paths**, one for each risk factor, and let us visualize these.

In [ ]:
path_no = 0
paths1 = port.underlying_objects['gbm'].get_instrument_values()[:, path_no]
paths2 = port.underlying_objects['jd'].get_instrument_values()[:, path_no]

The plot illustrates that the two paths are indeed **highly positively correlated**. In this case, **no jump** occurs for the `jump_diffusion` object.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(port.time_grid, paths1, 'r', label='gbm')
plt.plot(port.time_grid, paths2, 'b', label='jd')
plt.gcf().autofmt_xdate()
plt.legend(loc=0); plt.grid(True)
# highly correlated underlyings
# -- with no (visible) jumps

**Copyright, License & Disclaimer**

© Dr. Yves J. Hilpisch | The Python Quants GmbH

DX Analytics (the "dx library" or "dx package") is licensed under the GNU Affero General
Public License version 3 or later (see http://www.gnu.org/licenses/).

DX Analytics comes with no representations or warranties, to the extent
permitted by applicable law.

http://tpq.io | [dx@tpq.io](mailto:team@tpq.io) |
http://twitter.com/dyjh

<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

**Quant Platform** | http://pqp.io

**Python for Finance Training** | http://training.tpq.io

**Certificate in Computational Finance** | http://compfinance.tpq.io

**Derivatives Analytics with Python (Wiley Finance)** |
http://dawp.tpq.io

**Python for Finance (2nd ed., O'Reilly)** |
http://py4fi.tpq.io